In [ ]:
%%bash
nvidia-smi --query-gpu=name,memory.total --format=csv 2> /dev/null || echo "Không có GPU: chạy bằng CPU"
nproc; free -g | head -2
df -h /tmp /kaggle/working

In [ ]:
%%bash
set -e
LOG=/tmp/pg_build.log; : > $LOG
# Toàn bộ output (kể cả warning) ghi vào $LOG; chỉ khi lỗi mới in 30 dòng cuối
run() { "$@" >> $LOG 2>&1 || { echo "LỖI khi chạy: $*"; tail -30 $LOG; exit 1; }; }

if [ -x /opt/pg12/bin/pg_config ] && [ -f "$(/opt/pg12/bin/pg_config --pkglibdir)/pg_hint_plan.so" ]; then
  echo "Đã build từ trước, bỏ qua."
else
  run apt-get update -qq
  run apt-get install -y -qq build-essential libreadline-dev zlib1g-dev

  # Xóa mã nguồn còn lại từ lần chạy trước (nếu không, git clone báo "already exists")
  cd /tmp
  rm -rf postgresql-12.5 postgresql-12.5.tar.gz pg_hint_plan
  run wget -q https://ftp.postgresql.org/pub/source/v12.5/postgresql-12.5.tar.gz
  tar xzf postgresql-12.5.tar.gz && cd postgresql-12.5
  run ./configure --prefix=/opt/pg12
  run make -j"$(nproc)"
  run make install

  cd /tmp
  run git -c advice.detachedHead=false clone -q -b REL12_1_3_7 https://github.com/ossc-db/pg_hint_plan.git
  cd pg_hint_plan
  run make PG_CONFIG=/opt/pg12/bin/pg_config
  run make PG_CONFIG=/opt/pg12/bin/pg_config install
fi

/opt/pg12/bin/postgres --version
# Prefix không chứa "postgres"/"pgsql" nên PG đặt extension trong lib/postgresql/: hỏi đúng chỗ qua pg_config
PKG=$(/opt/pg12/bin/pg_config --pkglibdir)
[ -f "$PKG/pg_hint_plan.so" ] || { echo "Không thấy pg_hint_plan.so trong $PKG"; exit 1; }
echo "pg_hint_plan OK"

In [ ]:
%%bash
set -e
id -u postgres > /dev/null 2>&1 || useradd -m -s /bin/bash postgres

if [ ! -f /tmp/pgdata/PG_VERSION ]; then
mkdir -p /tmp/pgdata && chown postgres /tmp/pgdata
su postgres -c "/opt/pg12/bin/initdb -D /tmp/pgdata -U postgres --auth=trust" > /dev/null

# Cấu hình theo conf/balsa-postgresql.conf của tác giả, giảm bộ nhớ cho vừa máy Kaggle
cat >> /tmp/pgdata/postgresql.conf <<'EOF'
listen_addresses = 'localhost'
shared_buffers = 8GB
temp_buffers = 32GB
work_mem = 512MB
effective_cache_size = 32GB
max_parallel_workers_per_gather = 4
max_parallel_maintenance_workers = 4
geqo = off
autovacuum = off
shared_preload_libraries = 'pg_hint_plan'
dynamic_shared_memory_type = mmap
EOF
fi

su postgres -c "/opt/pg12/bin/pg_ctl -D /tmp/pgdata status" > /dev/null \
  || su postgres -c "/opt/pg12/bin/pg_ctl -D /tmp/pgdata -l /tmp/pg.log start -w"
/opt/pg12/bin/psql -U postgres -h localhost -c "SHOW shared_buffers;"

In [ ]:
%%bash
set -e
export PATH=/opt/pg12/bin:$PATH PGHOST=localhost PGUSER=postgres

# Phiên Kaggle mới thì /opt và /tmp bị xóa sạch: phải chạy lại ô 2 và ô 3 trước
[ -x /opt/pg12/bin/psql ] || { echo "Chưa có PostgreSQL ở /opt/pg12 (phiên Kaggle mới?): chạy lại ô 2 rồi ô 3."; exit 1; }
pg_isready -q || { echo "PostgreSQL chưa chạy: chạy lại ô 3."; exit 1; }
# Lần chạy trước vẫn đang nạp (vd. trang bị tải lại giữa chừng): không được xóa CSDL đang nạp dở
if pgrep -f load_job_postgres.sh > /dev/null; then
  echo "Lần chạy trước vẫn đang nạp. Xem tiến độ: tail /tmp/load.log. Đợi xong rồi chạy lại ô 4."; exit 1
fi

# Mã nguồn của tác giả: script nạp IMDB, code HybridQO và các split
[ -d /tmp/lqo_ml_perspective ] || git clone -q https://github.com/edualc/lqo_ml_perspective.git /tmp/lqo_ml_perspective

if [ -f /tmp/imdb_loaded ]; then
  echo "IMDB đã nạp từ trước, bỏ qua."
else
  # Dọn phần dở dang của lần chạy trước
  dropdb --if-exists imdbload
  rm -rf /tmp/imdb && mkdir -p /tmp/imdb

  # Dữ liệu IMDB của JOB (link cũ homepages.cwi.nl/~boncz/job đã 404); giữ file nén để lần sau khỏi tải lại
  wget -q -c -O /tmp/imdb.tgz https://event.cwi.nl/da/job/imdb.tgz
  tar xzf /tmp/imdb.tgz -C /tmp/imdb

  # Thêm dòng tiêu đề cột vào các file CSV (script của tác giả)
  pip install -q absl-py
  python /tmp/lqo_ml_perspective/data/prepend_imdb_headers.py --csv_dir /tmp/imdb 2> /dev/null
  chmod -R a+rX /tmp/imdb /tmp/lqo_ml_perspective

  # Nạp vào PostgreSQL: tạo bảng, nạp CSV, tạo index, ANALYZE (script của tác giả)
  echo "Đang nạp, khoảng 30–45 phút; log ở /tmp/load.log ..."
  su postgres -c "PATH=/opt/pg12/bin:\$PATH PGHOST=localhost bash /tmp/lqo_ml_perspective/balsa/load-postgres/load_job_postgres.sh /tmp/imdb" > /tmp/load.log 2>&1 &
  PID=$!
  # In tiến độ mỗi phút: mỗi bảng nạp xong, psql ghi một dòng "COPY <số dòng>" vào log
  while kill -0 $PID 2> /dev/null; do
    sleep 60
    N=$(grep -c '^COPY ' /tmp/load.log || true)
    if grep -q 'ANALYZE VERBOSE' /tmp/load.log; then STEP="ANALYZE (bước cuối)"
    elif grep -q add_fks /tmp/load.log; then STEP="tạo khóa ngoại"
    else STEP="nạp CSV"; fi
    echo "$(date +%H:%M)  đã nạp $N/21 bảng, đang: $STEP"
  done
  wait $PID || { echo "Script nạp dừng vì lỗi:"; tail -20 /tmp/load.log; exit 1; }

  # Script chạy các lệnh \copy ở nền nên lỗi của chúng không làm dừng script: phải tự tìm trong log
  if grep -q ERROR /tmp/load.log; then
    echo "Có lỗi khi nạp:"; grep ERROR /tmp/load.log | head -10; exit 1
  fi
fi

# Đếm chính xác (count(*)) số dòng của cả 21 bảng, so với số chuẩn của IMDB bản JOB (khoảng 1–2 phút).
# ON_ERROR_STOP để lỗi SQL làm ô dừng thật (mặc định psql vẫn trả mã 0 khi câu lệnh lỗi).
BAD=""
while read -r t n; do
  c=$(psql -d imdbload -At -v ON_ERROR_STOP=1 -c "SELECT count(*) FROM $t" < /dev/null) \
    || { echo "Không đếm được bảng $t"; exit 1; }
  [ "$c" = "$n" ] || BAD+="$t: có $c, cần $n"$'\n'
done <<'EOF'
aka_name 901343
aka_title 361472
cast_info 36244344
char_name 3140339
comp_cast_type 4
company_name 234997
company_type 4
complete_cast 135086
info_type 113
keyword 134170
kind_type 7
link_type 18
movie_companies 2609129
movie_info 14835720
movie_info_idx 1380035
movie_keyword 4523930
movie_link 29997
name 4167491
person_info 2963664
role_type 12
title 2528312
EOF
if [ -n "$BAD" ]; then
  echo "Các bảng sau sai số dòng:"; echo "$BAD"; exit 1
fi
echo "Đủ 21 bảng, số dòng khớp."

touch /tmp/imdb_loaded
rm -rf /tmp/imdb /tmp/imdb.tgz   # xóa CSV và file nén cho đỡ tốn chỗ

In [ ]:
%%bash
/opt/pg12/bin/psql -U postgres -h localhost -d imdbload -c "/*+ MergeJoin(t mc) */ EXPLAIN SELECT count(*) FROM title t, movie_companies mc WHERE t.id = mc.movie_id;" | grep -i "join"

In [ ]:
%%bash
set -e
pip install -q uv
[ -x /tmp/hqo_env/bin/python ] || uv venv -q -p 3.10 /tmp/hqo_env   # chạy lại thì giữ môi trường cũ
uv pip install -q --python /tmp/hqo_env/bin/python torch==1.13.1 --index-url https://download.pytorch.org/whl/cu117
uv pip install -q --python /tmp/hqo_env/bin/python "numpy<2" psqlparse==1.0rc7 torchfold psycopg2-binary wandb tqdm
/tmp/hqo_env/bin/python -c "import torch, psqlparse, torchfold; print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'chay bang CPU')"

In [ ]:
import pathlib, re, subprocess

SPLIT = "base_query_split_1"   # leave_one_out_split_1 đã chạy xong (hybridqo_log/)
REP = 1                           # lần chạy thứ mấy; tên file kết quả là hybrid_qo__JOB__<SPLIT>__<REP>.csv như tác giả
N_EPOCHS = 11                     # epoch 0..10: test ở epoch 0 và 10, so được với mốc epoch 10 của tác giả (xem "Vì sao 11 epoch")

hq = pathlib.Path("/tmp/lqo_ml_perspective/hybrid_qo")
PATCHED = ["ImportantConfig.py", "run_mcts.py", "torchfold.py", "mcts.py"]

# 0) Khôi phục bản gốc để không vá chồng lên lần chạy trước
subprocess.run(["git", "-C", str(hq.parent), "checkout", "--"] + [f"hybrid_qo/{f}" for f in PATCHED], check=True)

# 1) ImportantConfig.py: lấy 5 dòng cấu hình JOB đang bị comment, chèn trước dòng tính mcts_input_size
cfg = hq / "ImportantConfig.py"
lines = cfg.read_text().splitlines()
start = next(i for i, l in enumerate(lines) if l.strip() == "# JOB")
job = [l.strip().replace("# self.", "self.", 1) for l in lines[start:start + 8] if l.strip().startswith("# self.")]
assert len(job) == 5, job
extra = job + [f"self.n_epochs = {N_EPOCHS}",
               f"self.queries_file = 'workload/JOB__{SPLIT}__train.json'",
               "self.ip = 'localhost'"]
idx = next(i for i, l in enumerate(lines) if "self.mcts_input_size" in l)
indent = re.match(r"\s*", lines[idx]).group()
lines[idx:idx] = [indent + l for l in extra]
cfg.write_text("\n".join(lines) + "\n")

# 2) run_mcts.py: dừng train khi hết ngân sách thời gian (biến môi trường HQO_BUDGET_H, đơn vị giờ)
run = hq / "run_mcts.py"
s = run.read_text()
loop = "    for epoch in tqdm(range(config.n_epochs), total=config.n_epochs, desc='Iterating over epochs...'):\n"
guard = ("        if time.time() - _START > float(os.environ.get('HQO_BUDGET_H', '8.5')) * 3600:\n"
         "            print('Het ngan sach thoi gian, dung train o epoch', epoch, flush=True)\n"
         "            break\n")
assert loop in s
if guard not in s:
    s = s.replace("import random\n", "import random\nimport os, time\n_START = time.time()\n", 1)
    s = s.replace(loop, loop + guard, 1)

# 3) Không test lại nếu epoch cuối vừa test (giữ epoch 10 đúng 113 dòng như log tác giả)
in_loop = "            test_epoch(hinter, test_queries, epoch, query_log_file_path)\n"
final = "    # Final eval\n    test_epoch(hinter, test_queries, epoch, query_log_file_path)\n"
assert in_loop in s and final in s
s = s.replace(loop, "    _last_test = None\n" + loop, 1)
s = s.replace(in_loop, in_loop + "            _last_test = epoch\n", 1)
s = s.replace(final, "    # Final eval\n    if _last_test != epoch:\n"
                     "        test_epoch(hinter, test_queries, epoch, query_log_file_path)\n", 1)
run.write_text(s)

# 4) Chỉ dùng CUDA khi có GPU (code gốc gọi cứng cuda, máy CPU sẽ lỗi)
tf = hq / "torchfold.py"
t = tf.read_text()
assert t.count("        self._cuda = cuda\n") == 2
tf.write_text(t.replace("        self._cuda = cuda\n", "        self._cuda = cuda and torch.cuda.is_available()\n")
               .replace("        self._cuda = True\n", "        self._cuda = torch.cuda.is_available()\n"))
mc = hq / "mcts.py"
m = mc.read_text()
assert m.count("        predictionNet.cuda()\n") == 2
mc.write_text(m.replace("        predictionNet.cuda()\n", "        predictionNet.to(config.device)\n"))

# Cho các ô %%bash phía sau biết split và lần chạy
import os
os.environ["HQO_SPLIT"], os.environ["HQO_REP"] = SPLIT, str(REP)

print("\n".join(extra))
print("Kết quả sẽ lưu thành:", f"hybrid_qo__JOB__{SPLIT}__{REP}.csv")

In [ ]:
%%bash
[ -x /tmp/hqo_env/bin/python ] || { echo "Chưa có môi trường Python: notebook thiếu ô 5 hoặc ô 5 lỗi"; exit 1; }
[ -n "$HQO_SPLIT" ] || { echo "Chưa chạy ô 6"; exit 1; }
cd /tmp/lqo_ml_perspective/hybrid_qo
mkdir -p model /kaggle/working/hqo_logs
rm -rf logs && ln -s /kaggle/working/hqo_logs logs   # log ghi thẳng vào thư mục được giữ lại
OUT="/kaggle/working/run_mcts__${HQO_SPLIT}__${HQO_REP}.out"

export WANDB_MODE=disabled      # không cần tài khoản Weights & Biases
export HQO_BUDGET_H=10          # chốt an toàn; bình thường 11 epoch xong trước (6,5–9,5 giờ)
touch /tmp/hqo_start
/tmp/hqo_env/bin/python run_mcts.py > "$OUT" 2>&1 &
PID=$!
echo "Bắt đầu $(date +%H:%M), split $HQO_SPLIT, lần chạy $HQO_REP. Output đầy đủ: $OUT"

# Đợi run_mcts.py tạo file query_log của lần chạy này
LOG=""
while [ -z "$LOG" ] && kill -0 $PID 2> /dev/null; do
  sleep 5
  LOG=$(find /kaggle/working/hqo_logs -name '*__query_log.csv' -newer /tmp/hqo_start | head -1)
done

# In từng dòng log khi được ghi; tail tự dừng khi run_mcts.py kết thúc
if [ -n "$LOG" ]; then
  tail -n +2 -f --pid=$PID "$LOG" | awk -F, '
    function summary() {
      if (n) printf "=== %s epoch %s xong lúc %s: %d câu | PG %.1fs | HybridQO %.1fs | speedup %.3f | thắng/hòa/thua %d/%d/%d ===\n",
                    kind, ep, now(), n, spg, sh, (sh > 0 ? spg / sh : 0), w, t, l
      n = spg = sh = w = t = l = 0
    }
    function now(  d) { "date +%H:%M" | getline d; close("date +%H:%M"); return d }
    {
      k = ($2 == "1") ? "test" : "train"
      if ($1 != ep || k != kind) { summary(); ep = $1; kind = k }
      r = ($9 > 0) ? $5 / $9 : 0
      n++; spg += $5; sh += $9
      if (r > 1.1) w++; else if (r < 0.9) l++; else t++
      printf "ep %2s %-5s %-4s  PG %8.3fs  HybridQO %8.3fs  x%.2f\n", $1, k, $3, $5, $9, r
      fflush()
    }
    END { summary() }'
fi

wait $PID; RC=$?
grep -h "Het ngan sach" "$OUT" || true
show_error() {
  echo "run_mcts.py dừng vì lỗi (mã $RC). 30 dòng cuối của $OUT:"
  tail -30 "$OUT" | tr '\r' '\n' | grep -v "it/s" | tail -30
  exit 1
}
[ -n "$LOG" ] || show_error

# Lưu kết quả theo định dạng wandb_export của tác giả (code giống kaggle/to_wandb_export.py).
# Nếu run_mcts.py lỗi giữa chừng thì vẫn lưu phần đã chạy, rồi mới báo lỗi.
/tmp/hqo_env/bin/python - "$LOG" <<'PY'
import csv, os, sys
METRICS = [("hinter_latency", "hinter_latency"), ("test_query", "test_query"), ("epoch", "epoch"),
           ("pg_lateny", "pg_latency"), ("pg_plan_time", "pg_plan_time"), ("mcts_time", "mcts_time"),
           ("hinter_plan_time", "hinter_plan_time"), ("MPHE_time", "MPHE_time")]
query_log, split, rep = sys.argv[1], os.environ["HQO_SPLIT"], os.environ["HQO_REP"]
run = os.path.basename(query_log).replace("__query_log.csv", "")
rows = list(csv.DictReader(open(query_log)))
out_dir = "/kaggle/working/wandb_export"
os.makedirs(out_dir, exist_ok=True)
out = os.path.join(out_dir, f"hybrid_qo__JOB__{split}__{rep}.csv")
with open(out, "w", newline="") as f:
    w = csv.writer(f, quoting=csv.QUOTE_ALL)
    w.writerow(["Step"] + [f"{run} - {m}{s}" for m, _ in METRICS for s in ("", "__MIN", "__MAX")])
    for step, r in enumerate(rows):
        w.writerow([step] + [r[c] for _, c in METRICS for _ in range(3)])
per_epoch = {}
for r in rows:
    per_epoch[int(r["epoch"])] = per_epoch.get(int(r["epoch"]), 0) + 1
full = [e for e, n in per_epoch.items() if n == 113 and e <= 50]
print(f"Đã lưu {out}: {len(rows)} dòng; epoch tác giả sẽ chọn (đủ 113 dòng): {max(full) if full else 'KHÔNG CÓ'}")
PY
[ $RC -eq 0 ] || show_error
echo "Xong lúc $(date +%H:%M)."